# Angular 18 — Complete Instructor Reference Guide
**Release Date:** May 22, 2024 | **Codename:** N/A | **Era:** Signals Era — Stable APIs

---

## Release at a Glance

| Requirement | Version |
|---|---|
| Node.js | 18.19+ or 20.11+ or 22+ |
| TypeScript | 5.4+ |
| RxJS | 7.4+ |
| Zone.js | 0.14.x (optional in v18) |

### Upgrade Command
```bash
ng update @angular/core@18 @angular/cli@18 @angular/material@18
```

---

## Top Features in Angular 18

| # | Feature | Status | Impact |
|---|---|---|---|
| 1 | **Zoneless Change Detection** | Experimental | Eliminates Zone.js dependency |
| 2 | **Stable Signals** — `signal()`, `computed()`, `effect()` | **Stable** | Reactive primitives production-ready |
| 3 | **Signal-based `output()`** | Developer Preview | Replaces `@Output` + EventEmitter |
| 4 | **`linkedSignal()`** | Developer Preview | Writable derived signal |
| 5 | **`@let` Template Variable** | Developer Preview | Local template variable declarations |
| 6 | **Material 3 (M3) Components** | **Stable** | Modern Material Design UI |
| 7 | **Incremental Hydration** | Developer Preview | Finer-grained SSR hydration |
| 8 | **Route-level Render Mode** | Developer Preview | Per-route SSR/CSR/Prerender control |
| 9 | **`input.required()` stable** | **Stable** | Required signal inputs fully stable |
| 10 | **TypeScript 5.4 Support** | **Stable** | Latest TS features |

---

## Why Angular 18 Matters

Angular 18 is the **signals stabilization release** — after the signals developer preview in v16 and signal inputs in v17, v18 promotes the core signal APIs (`signal`, `computed`, `effect`) to **stable** status. It also introduces **Zoneless Change Detection** as an experimental opt-in, marking the beginning of Zone.js's eventual deprecation. Material 3 components are now stable, giving teams a production-ready modern UI library.

# Section 1 — Detailed Notes

---

## 1. Stable Signal APIs: `signal()`, `computed()`, `effect()`

### What Changed
In Angular 16, Signals were introduced as a Developer Preview. In Angular 18, the core trio — `signal()`, `computed()`, and `effect()` — are promoted to **stable**. This means they are safe for production use with a commitment to no breaking API changes.

### Core Concepts

| API | Purpose | Description |
|---|---|---|
| `signal(value)` | Writable reactive state | Creates a reactive value container |
| `computed(fn)` | Derived read-only state | Lazily recalculates when dependencies change |
| `effect(fn)` | Side effects on signal changes | Runs whenever tracked signals change |

### Behavior Details
- **`signal()`**: Use `.set()`, `.update()`, `.mutate()` to change values
- **`computed()`**: Read-only. Memoized — only re-runs when dependencies change
- **`effect()`**: Automatically tracks signal reads inside the function. Cleaned up when component is destroyed
- **Change Detection**: Signal changes trigger targeted re-renders without zone.js dirty-checking the entire component tree

```typescript
import { signal, computed, effect } from '@angular/core';

// Writable signal
const count = signal(0);

// Derived signal (read-only)
const doubled = computed(() => count() * 2);

// Side effect
effect(() => {
  console.log(`Count changed to: ${count()}`);
});

count.set(5);     // logs "Count changed to: 5"
count.update(v => v + 1);  // logs "Count changed to: 6"
console.log(doubled());    // 12
```

---

## 2. Zoneless Change Detection — `provideExperimentalZonelessChangeDetection()`

### What It Is
An **experimental** opt-in that completely removes Zone.js from the Angular application. Without Zone.js, Angular only re-renders when a signal changes, an async pipe emits, or manual `markForCheck()` / `ChangeDetectorRef.detectChanges()` is called.

### Why It Matters
- **Smaller bundle**: Zone.js is ~13KB (minified+gzipped) — removing it shrinks your bundle
- **Predictable rendering**: No more accidental re-renders from microtask patching
- **Performance**: Only the components that actually changed are checked
- **Future-proof**: Zoneless is the long-term direction for Angular

### Setup
```typescript
// main.ts
import { bootstrapApplication } from '@angular/platform-browser';
import { provideExperimentalZonelessChangeDetection } from '@angular/core';
import { AppComponent } from './app/app.component';

bootstrapApplication(AppComponent, {
  providers: [
    provideExperimentalZonelessChangeDetection()
  ]
});
```

```json
// angular.json — remove zone.js polyfill
{
  "polyfills": []  // Remove "zone.js" from this array
}
```

### Rules for Zoneless Components
- Use **Signals** for reactive state
- Use **`async` pipe** for Observables in templates
- Use `ChangeDetectorRef.markForCheck()` for manual triggering
- Avoid imperative DOM manipulation outside Angular context

### Comparison: Zone.js vs Zoneless

| Aspect | Zone.js (Default) | Zoneless (Experimental) |
|---|---|---|
| Bundle size | +13KB | -13KB |
| Change detection trigger | Any async operation | Signal change / manual |
| Re-render scope | Entire component tree | Only changed components |
| Compatibility | All legacy code | Requires signal-aware code |

---

## 3. Signal-based `output()` — Developer Preview

### What It Is
A new function-based API for declaring component outputs, replacing the `@Output()` decorator + `EventEmitter` pattern.

### Before vs After
```typescript
// BEFORE (Angular 17 and earlier)
import { Output, EventEmitter } from '@angular/core';

@Component({...})
export class ButtonComponent {
  @Output() clicked = new EventEmitter<string>();
  
  onClick() {
    this.clicked.emit('button clicked');
  }
}
```

```typescript
// AFTER (Angular 18+)
import { output } from '@angular/core';

@Component({...})
export class ButtonComponent {
  clicked = output<string>();
  
  onClick() {
    this.clicked.emit('button clicked');
  }
}
```

### Key Differences
- **No `EventEmitter`** — `output()` returns an `OutputEmitterRef`
- **Type-safe** — generic type is explicit: `output<string>()`
- **Simpler subscription** — in tests: `outputRef.subscribe(value => ...)`
- **Required output** — Not applicable (outputs are always optional)
- **Alias** — `output({ alias: 'buttonClick' })`

### Parent Component Usage
```html
<!-- Template usage is identical — no change needed -->
<app-button (clicked)="handleClick($event)" />
```

---

## 4. `linkedSignal()` — Developer Preview

### What It Is
A **writable computed signal** — a signal whose default value is derived from another signal, but can also be manually overridden. When the source signal changes, the linked signal resets to the new computed value.

### Use Case
Ideal for UI state that should follow data changes but can also be modified by the user (e.g., a selected item that resets when the list changes).

```typescript
import { signal, linkedSignal } from '@angular/core';

const items = signal(['Apple', 'Banana', 'Cherry']);

// selectedItem follows items[0] by default, but can be manually set
const selectedItem = linkedSignal(() => items()[0]);

console.log(selectedItem()); // 'Apple'

selectedItem.set('Banana'); // manual override
console.log(selectedItem()); // 'Banana'

items.set(['Dog', 'Cat', 'Fish']); // source changes → resets
console.log(selectedItem()); // 'Dog' (reset to new computed value)
```

### Comparison with `computed()`

| Feature | `computed()` | `linkedSignal()` |
|---|---|---|
| Writable | ❌ No | ✅ Yes |
| Resets on source change | ✅ Always | ✅ Yes |
| Manual override | ❌ No | ✅ Yes |
| Use case | Derived read-only data | UI selection state |

---

## 5. `@let` Template Variable Declarations — Developer Preview

### What It Is
A new template syntax that allows declaring **local variables** inside Angular templates without needing `*ngIf as` workarounds.

### Before vs After
```html
<!-- BEFORE: Workaround using *ngIf as -->
<ng-container *ngIf="user$ | async as user">
  <h1>{{ user.name }}</h1>
  <p>{{ user.email }}</p>
</ng-container>

<!-- BEFORE: Repeated async pipe (bad practice) -->
<h1>{{ (user$ | async)?.name }}</h1>
<p>{{ (user$ | async)?.email }}</p>
```

```html
<!-- AFTER: Clean @let syntax -->
@let user = user$ | async;
@if (user) {
  <h1>{{ user.name }}</h1>
  <p>{{ user.email }}</p>
}

<!-- Also works for computed expressions -->
@let fullName = user.firstName + ' ' + user.lastName;
@let total = items.length * price;
<p>{{ fullName }} — Total: {{ total | currency }}</p>
```

### Rules
- `@let` is **block-scoped** to the template block it's declared in
- Can reference signals, pipes, expressions, and method calls
- Cannot be reassigned (read-only after declaration)

---

## 6. Material 3 (M3) Components — Stable

### What Changed
Angular Material's Material Design 3 (M3) components are now **stable** after being in experimental status since Angular 17.

### Key M3 Changes vs M2

| Aspect | Material 2 (M2) | Material 3 (M3) |
|---|---|---|
| Color system | Palette-based | Dynamic color roles (primary, secondary, tertiary) |
| Shape | Rounded corners fixed | Expressive shape scale |
| Typography | Roboto-centric | Flexible type scale |
| Component names | `mat-button` | `mat-button` (same) |
| Theme setup | `mat.define-light-theme()` | `mat.define-theme()` |

### Setup with M3 Theme
```scss
// styles.scss
@use '@angular/material' as mat;

$theme: mat.define-theme((
  color: (
    theme-type: light,
    primary: mat.$violet-palette,
    tertiary: mat.$orange-palette,
  ),
  typography: (
    brand-family: 'Inter, sans-serif',
  ),
  density: (
    scale: 0,
  )
));

html {
  @include mat.all-component-themes($theme);
}
```

---

## 7. Incremental Hydration — Developer Preview

### What It Is
Extends Angular's Non-Destructive Hydration (stable since v17) to allow **lazy, trigger-based hydration** of specific component subtrees. Server renders the full HTML; the browser hydrates only the parts that are needed based on triggers.

### Triggers
```typescript
// app.config.ts
import { provideClientHydration, withIncrementalHydration } from '@angular/platform-browser';

bootstrapApplication(AppComponent, {
  providers: [
    provideClientHydration(withIncrementalHydration())
  ]
});
```

```html
<!-- Component template — hydrate on interaction -->
@defer (hydrate on interaction) {
  <app-comments />
}

@defer (hydrate on viewport) {
  <app-product-recommendations />
}

@defer (hydrate on idle) {
  <app-footer-links />
}
```

### Impact
- Server sends full HTML → fast FCP
- JS for heavy components not parsed until needed → faster TTI
- Works alongside existing `@defer` loading triggers

---

## 8. Route-level Render Mode — Developer Preview

### What It Is
Allows configuring **per-route rendering strategy** — Server-Side Rendering (SSR), Client-Side Rendering (CSR), or Prerendering — directly in the route configuration.

```typescript
// app.routes.server.ts
import { RenderMode, ServerRoute } from '@angular/ssr';

export const serverRoutes: ServerRoute[] = [
  { path: 'home', renderMode: RenderMode.Prerender },
  { path: 'products', renderMode: RenderMode.Server },
  { path: 'dashboard', renderMode: RenderMode.Client },
  { path: 'products/:id', renderMode: RenderMode.Prerender,
    async getPrerenderParams() {
      return [{ id: '1' }, { id: '2' }, { id: '3' }];
    }
  },
];
```

| Mode | When to Use |
|---|---|
| `Prerender` | Static content, blog posts, product listings |
| `Server` | Dynamic, personalized pages (auth-protected) |
| `Client` | Highly interactive, dashboard-style pages |

---

## 9. TypeScript 5.4 Support

### New TypeScript 5.4 Features Relevant to Angular

| Feature | Description |
|---|---|
| `NoInfer<T>` utility type | Prevents widening of inferred type parameters |
| Preserved narrowing in closures | Type narrowing preserved after last assignment |
| `Object.groupBy` / `Map.groupBy` typings | New built-in grouping APIs typed |

```typescript
// NoInfer<T> example — useful for Angular service type constraints
function createService<T>(defaultValue: T, fallback: NoInfer<T>): T {
  return defaultValue ?? fallback;
}
```

# Section 2 — Code Examples

---

## Example 1: Full Signal-based Counter Component (Stable APIs)

```typescript
// counter.component.ts
import { Component, signal, computed, effect } from '@angular/core';
import { CommonModule } from '@angular/common';

@Component({
  selector: 'app-counter',
  standalone: true,
  imports: [CommonModule],
  template: `
    <div class="counter-card">
      <h2>Signal Counter (Angular 18 Stable)</h2>
      <p>Count: <strong>{{ count() }}</strong></p>
      <p>Doubled: <strong>{{ doubled() }}</strong></p>
      <p>Is Even: <strong>{{ isEven() }}</strong></p>
      <p>History: {{ history().join(', ') }}</p>

      <div class="actions">
        <button (click)="decrement()">-</button>
        <button (click)="reset()">Reset</button>
        <button (click)="increment()">+</button>
      </div>
    </div>
  `
})
export class CounterComponent {
  count = signal(0);
  doubled = computed(() => this.count() * 2);
  isEven = computed(() => this.count() % 2 === 0 ? 'Yes' : 'No');
  history = signal<number[]>([]);

  constructor() {
    // effect() is now stable — runs whenever count changes
    effect(() => {
      this.history.update(h => [...h.slice(-4), this.count()]);
    });
  }

  increment() { this.count.update(v => v + 1); }
  decrement() { this.count.update(v => v - 1); }
  reset()     { this.count.set(0); }
}
```

---

## Example 2: Zoneless Application Setup

```typescript
// main.ts — Zoneless Angular 18 app
import { bootstrapApplication } from '@angular/platform-browser';
import { provideExperimentalZonelessChangeDetection } from '@angular/core';
import { provideRouter } from '@angular/router';
import { AppComponent } from './app/app.component';
import { routes } from './app/app.routes';

bootstrapApplication(AppComponent, {
  providers: [
    provideExperimentalZonelessChangeDetection(),  // Removes Zone.js
    provideRouter(routes),
  ]
});
```

```typescript
// app.component.ts — Zoneless-compatible component
import { Component, signal, inject } from '@angular/core';
import { HttpClient } from '@angular/common/http';
import { toSignal } from '@angular/core/rxjs-interop';

@Component({
  selector: 'app-root',
  standalone: true,
  template: `
    <h1>{{ title() }}</h1>
    @if (user()) {
      <p>Welcome, {{ user()!.name }}</p>
    } @else {
      <p>Loading...</p>
    }
  `
})
export class AppComponent {
  private http = inject(HttpClient);
  title = signal('Zoneless Angular 18 App');

  // toSignal converts Observable → Signal (no async pipe needed)
  user = toSignal(
    this.http.get<{ name: string }>('/api/user')
  );
}
```

---

## Example 3: `output()` — New Signal-based Output API

```typescript
// rating.component.ts
import { Component, input, output, signal, computed } from '@angular/core';

@Component({
  selector: 'app-rating',
  standalone: true,
  template: `
    <div class="rating">
      <h3>{{ label() }}</h3>
      @for (star of stars; track star) {
        <span
          [class.filled]="star <= currentRating()"
          (click)="selectStar(star)"
          (mouseenter)="hover.set(star)"
          (mouseleave)="hover.set(0)">
          ★
        </span>
      }
      <p>Rating: {{ currentRating() }} / {{ maxStars() }}</p>
      <button (click)="submitRating()" [disabled]="currentRating() === 0">
        Submit
      </button>
    </div>
  `
})
export class RatingComponent {
  // Signal inputs (stable in v18)
  label = input('Rate this product');
  maxStars = input(5);

  // New output() API (developer preview in v18)
  ratingSubmitted = output<number>();
  ratingChanged   = output<number>();

  hover         = signal(0);
  selectedStar  = signal(0);
  stars         = [1, 2, 3, 4, 5];

  currentRating = computed(() => this.hover() || this.selectedStar());

  selectStar(star: number) {
    this.selectedStar.set(star);
    this.ratingChanged.emit(star);   // emit via output()
  }

  submitRating() {
    this.ratingSubmitted.emit(this.currentRating());  // emit via output()
  }
}

// parent.component.ts — usage
@Component({
  selector: 'app-parent',
  standalone: true,
  imports: [RatingComponent],
  template: `
    <app-rating
      label="How was your experience?"
      [maxStars]="5"
      (ratingChanged)="onRatingChange($event)"
      (ratingSubmitted)="onSubmit($event)" />
  `
})
export class ParentComponent {
  onRatingChange(rating: number) { console.log('Changed to:', rating); }
  onSubmit(rating: number)       { console.log('Submitted:', rating); }
}
```

---

## Example 4: `linkedSignal()` — Smart Selection State

```typescript
// product-list.component.ts
import { Component, signal, linkedSignal, computed, inject } from '@angular/core';
import { ProductService } from './product.service';

interface Product { id: number; name: string; price: number; category: string; }

@Component({
  selector: 'app-product-list',
  standalone: true,
  template: `
    <div class="product-browser">
      <div class="filters">
        <label>Category:</label>
        @for (cat of categories(); track cat) {
          <button
            [class.active]="selectedCategory() === cat"
            (click)="selectedCategory.set(cat)">{{ cat }}</button>
        }
      </div>

      <div class="products">
        @for (product of filteredProducts(); track product.id) {
          <div
            class="product-card"
            [class.selected]="selectedProduct()?.id === product.id"
            (click)="selectedProduct.set(product)">
            <h3>{{ product.name }}</h3>
            <p>{{ product.price | currency }}</p>
          </div>
        }
      </div>

      @if (selectedProduct()) {
        <div class="product-detail">
          <h2>{{ selectedProduct()!.name }}</h2>
          <p>Price: {{ selectedProduct()!.price | currency }}</p>
        </div>
      }
    </div>
  `
})
export class ProductListComponent {
  private productService = inject(ProductService);

  allProducts = signal<Product[]>(this.productService.getAll());
  selectedCategory = signal('All');

  categories = computed(() => [
    'All',
    ...new Set(this.allProducts().map(p => p.category))
  ]);

  filteredProducts = computed(() => {
    const cat = this.selectedCategory();
    return cat === 'All'
      ? this.allProducts()
      : this.allProducts().filter(p => p.category === cat);
  });

  // linkedSignal: auto-selects first product when filtered list changes
  // but can be manually overridden by clicking a product card
  selectedProduct = linkedSignal<Product | null>(() => this.filteredProducts()[0] ?? null);
}
```

---

## Example 5: `@let` Template Variable in Action

```typescript
// user-profile.component.ts
import { Component, inject } from '@angular/core';
import { AsyncPipe, CurrencyPipe, DatePipe } from '@angular/common';
import { UserService } from './user.service';

@Component({
  selector: 'app-user-profile',
  standalone: true,
  imports: [AsyncPipe, CurrencyPipe, DatePipe],
  template: `
    @let user = currentUser$ | async;
    @let orders = userOrders$ | async;

    @if (user) {
      <!-- @let for computed expressions -->
      @let fullName = user.firstName + ' ' + user.lastName;
      @let initials = user.firstName[0] + user.lastName[0];
      @let isPremium = user.subscriptionTier === 'premium';
      @let orderCount = orders?.length ?? 0;
      @let totalSpent = orders?.reduce((sum, o) => sum + o.total, 0) ?? 0;

      <div class="profile-card" [class.premium]="isPremium">
        <div class="avatar">{{ initials }}</div>
        <h1>{{ fullName }}</h1>
        <p>Email: {{ user.email }}</p>
        <span class="badge" [class.gold]="isPremium">
          {{ isPremium ? 'Premium Member' : 'Standard Member' }}
        </span>

        <div class="stats">
          <div class="stat">
            <label>Total Orders</label>
            <strong>{{ orderCount }}</strong>
          </div>
          <div class="stat">
            <label>Total Spent</label>
            <strong>{{ totalSpent | currency }}</strong>
          </div>
          <div class="stat">
            <label>Member Since</label>
            <strong>{{ user.joinedAt | date:'mediumDate' }}</strong>
          </div>
        </div>

        <!-- Nested @let scoped to @if block -->
        @if (orderCount > 0) {
          @let lastOrder = orders![0];
          <p>Last order: {{ lastOrder.id }} on {{ lastOrder.date | date }}</p>
        }
      </div>
    } @else {
      <p>Loading profile...</p>
    }
  `
})
export class UserProfileComponent {
  private userService = inject(UserService);
  currentUser$ = this.userService.getCurrentUser();
  userOrders$  = this.userService.getUserOrders();
}
```

---

## Example 6: Incremental Hydration with `@defer`

```typescript
// app.config.ts — Enable incremental hydration
import { ApplicationConfig } from '@angular/core';
import { provideRouter } from '@angular/router';
import { provideClientHydration, withIncrementalHydration } from '@angular/platform-browser';
import { provideHttpClient, withFetch } from '@angular/common/http';

export const appConfig: ApplicationConfig = {
  providers: [
    provideRouter([]),
    provideClientHydration(withIncrementalHydration()),
    provideHttpClient(withFetch()),
  ]
};
```

```typescript
// product-page.component.ts
import { Component, input, signal, inject } from '@angular/core';
import { ProductService } from './product.service';
import { toSignal } from '@angular/core/rxjs-interop';

@Component({
  selector: 'app-product-page',
  standalone: true,
  template: `
    <!-- Hydrated immediately (above the fold) -->
    <app-product-hero [productId]="productId()" />

    <!-- Hydrated only when user scrolls into view -->
    @defer (hydrate on viewport) {
      <app-product-specs [productId]="productId()" />
    }

    <!-- Hydrated only when user clicks or hovers -->
    @defer (hydrate on interaction) {
      <app-product-reviews [productId]="productId()" />
    }

    <!-- Hydrated only when browser is idle -->
    @defer (hydrate on idle) {
      <app-related-products [productId]="productId()" />
      <app-recently-viewed />
    }

    <!-- Loading placeholder visible until hydrated -->
    @defer (hydrate on viewport) {
      <app-size-guide />
    } @loading {
      <div class="skeleton-loader">Loading size guide...</div>
    }
  `
})
export class ProductPageComponent {
  productId = input.required<string>();
}
```

---

## Example 7: Route-level Render Mode + Material 3

```typescript
// app.routes.server.ts — Route-level render mode
import { RenderMode, ServerRoute } from '@angular/ssr';

export const serverRoutes: ServerRoute[] = [
  // Static pages — prerendered at build time
  { path: '',        renderMode: RenderMode.Prerender },
  { path: 'about',   renderMode: RenderMode.Prerender },
  { path: 'pricing', renderMode: RenderMode.Prerender },

  // Dynamic product pages — SSR on each request
  { path: 'products',     renderMode: RenderMode.Server },
  { path: 'products/:id', renderMode: RenderMode.Server },

  // Authenticated pages — CSR only (no SSR needed)
  { path: 'dashboard',  renderMode: RenderMode.Client },
  { path: 'account',    renderMode: RenderMode.Client },
  { path: 'checkout',   renderMode: RenderMode.Client },
];
```

```typescript
// Material 3 Theme + Component usage
// styles.scss
@use '@angular/material' as mat;

$app-theme: mat.define-theme((
  color: (
    theme-type: light,
    primary: mat.$indigo-palette,
    tertiary: mat.$pink-palette,
  ),
  density: (scale: -1)
));

html { @include mat.all-component-themes($app-theme); }
```

```html
<!-- M3 Component usage — same selectors, new design tokens -->
<mat-card appearance="outlined">
  <mat-card-header>
    <mat-card-title>Angular 18 Features</mat-card-title>
    <mat-card-subtitle>Material 3 — Stable</mat-card-subtitle>
  </mat-card-header>
  <mat-card-content>
    <mat-form-field appearance="outline">
      <mat-label>Search</mat-label>
      <input matInput placeholder="Type to search...">
    </mat-form-field>
  </mat-card-content>
  <mat-card-actions align="end">
    <button mat-button>Cancel</button>
    <button mat-flat-button color="primary">Submit</button>
  </mat-card-actions>
</mat-card>
```

# Section 3 — Use Cases

---

## Use Case 1: E-Commerce Platform — Zoneless + Signals for Performance

### Problem
A high-traffic e-commerce platform experiences sluggish cart and inventory pages. Zone.js triggers full component tree re-renders on every HTTP poll, setTimeout, and user interaction — causing janky UI on mobile devices.

### Angular 18 Solution
Migrate to **zoneless change detection** + **signal-based state** to limit re-renders to only the changed UI regions.

```typescript
// cart.store.ts — Signal-based store
import { Injectable, signal, computed } from '@angular/core';

export interface CartItem { id: string; name: string; price: number; qty: number; }

@Injectable({ providedIn: 'root' })
export class CartStore {
  private items = signal<CartItem[]>([]);

  itemCount = computed(() => this.items().reduce((sum, i) => sum + i.qty, 0));
  subtotal  = computed(() => this.items().reduce((sum, i) => sum + i.price * i.qty, 0));
  tax       = computed(() => this.subtotal() * 0.08);
  total     = computed(() => this.subtotal() + this.tax());
  isEmpty   = computed(() => this.items().length === 0);
  allItems  = this.items.asReadonly();

  addItem(item: CartItem) {
    this.items.update(cart => {
      const existing = cart.find(i => i.id === item.id);
      if (existing) return cart.map(i => i.id === item.id ? { ...i, qty: i.qty + 1 } : i);
      return [...cart, item];
    });
  }

  removeItem(id: string) { this.items.update(cart => cart.filter(i => i.id !== id)); }
  clearCart() { this.items.set([]); }
}

// main.ts
bootstrapApplication(AppComponent, {
  providers: [provideExperimentalZonelessChangeDetection()]
});
```

### Result

| Metric | Before (Zone.js) | After (Zoneless + Signals) |
|---|---|---|
| Re-renders per action | 47 components | 2–3 components |
| Cart update latency | 180ms | 12ms |
| Mobile Lighthouse Score | 61 | 89 |
| Bundle size | +13KB zone.js | -13KB (removed) |

---

## Use Case 2: Healthcare Portal — `@let` Template Variables for Complex Forms

### Problem
A healthcare data-entry portal has multi-section patient forms. Doctors complained about slow form rendering and repeated `*ngIf as` workarounds making templates unreadable. Async pipe subscriptions were duplicated across the template.

### Angular 18 Solution
Use `@let` to cleanly declare local template variables, eliminating repetition and improving readability.

```html
@let patient = patientData$ | async;
@let isValid = form.valid;
@let isDirty  = form.dirty;
@let canSubmit = isValid && isDirty && !isSubmitting();

@if (patient) {
  @let fullName  = patient.firstName + ' ' + patient.lastName;
  @let age       = calculateAge(patient.dateOfBirth);
  @let riskLevel = age > 65 ? 'High' : age > 45 ? 'Medium' : 'Low';

  <div class="patient-header" [class.high-risk]="riskLevel === 'High'">
    <h1>{{ fullName }}</h1>
    <span class="badge risk-{{ riskLevel | lowercase }}">
      {{ riskLevel }} Risk — Age {{ age }}
    </span>
  </div>

  <button type="submit" [disabled]="!canSubmit">
    {{ isSubmitting() ? 'Saving...' : 'Save Patient Record' }}
  </button>
}
```

### Impact
- Template code reduced by **40%** (eliminated 12 `*ngIf as` blocks)
- Zero duplicate async subscriptions
- Templates became self-documenting — onboarding time for new developers reduced

---

## Use Case 3: News Media — Incremental Hydration for LCP/TTI Optimization

### Problem
A news website uses SSR but the entire page — including below-fold comments, related articles, and ad widgets — is eagerly hydrated, blocking Time-to-Interactive (TTI) and wasting CPU on content users may never scroll to.

### Angular 18 Solution
Use **incremental hydration** with `withIncrementalHydration()` and `@defer (hydrate on ...)` triggers.

```html
<!-- article-page.component.html -->

<!-- Immediately hydrated — critical above-fold content -->
<app-article-header [article]="article" />
<app-article-body   [content]="article.body" />

<!-- Hydrated when user scrolls to it -->
@defer (hydrate on viewport) {
  <app-related-articles [articleId]="article.id" />
}

<!-- Hydrated only when user interacts -->
@defer (hydrate on interaction) {
  <app-comments-section [articleId]="article.id" />
} @placeholder {
  <div class="comments-placeholder">💬 Load Comments ({{ article.commentCount }})</div>
}

<!-- Hydrated when browser is idle -->
@defer (hydrate on idle) {
  <app-newsletter-signup />
}
```

### Impact

| Metric | Before | After |
|---|---|---|
| JS parsed on load | 480KB | 95KB |
| Time to Interactive | 5.8s | 1.9s |
| Largest Contentful Paint | 2.4s | 1.1s |
| Comments hydrated for users who scroll | 100% | 38% (62% never needed) |

---

## Use Case 4: SaaS Dashboard — `linkedSignal()` for Smart Data Table

### Problem
A SaaS analytics dashboard has a data table where the selected row should auto-reset to the first row whenever the user changes filters (date range, region, product). Previously this required complex `ngOnChanges` logic and `BehaviorSubject` chains.

### Angular 18 Solution
`linkedSignal()` provides the perfect primitive: selection follows filter changes automatically but can be manually overridden by clicking a row.

```typescript
@Component({
  selector: 'app-analytics-table',
  standalone: true,
  template: `
    <div class="filters">
      <select (change)="region.set($any($event.target).value)">
        <option *ngFor="let r of regions" [value]="r">{{ r }}</option>
      </select>
    </div>

    <table>
      @for (row of tableData(); track row.id) {
        <tr
          [class.selected]="selectedRow()?.id === row.id"
          (click)="selectedRow.set(row)">
          <td>{{ row.label }}</td>
          <td>{{ row.value | number }}</td>
        </tr>
      }
    </table>

    @if (selectedRow()) {
      <app-row-detail [row]="selectedRow()!" />
    }
  `
})
export class AnalyticsTableComponent {
  region  = signal('All');
  regions = ['All', 'North', 'South', 'East', 'West'];

  tableData = computed(() => this.analyticsService.getByRegion(this.region()));

  // Auto-selects first row when region changes; user can click to override
  selectedRow = linkedSignal(() => this.tableData()[0] ?? null);

  constructor(private analyticsService: AnalyticsService) {}
}
```

### Impact
- Replaced 85 lines of `ngOnChanges` + `BehaviorSubject` logic with 1 `linkedSignal()` call
- Zero bugs from stale selection state after filter changes
- Reduced component complexity score by 60%

---

## Use Case 5: Fintech App — Route-level Render Mode for Compliance + Performance

### Problem
A fintech application has a mix of public marketing pages (SEO-critical), authenticated portfolio pages (no SEO needed, highly dynamic), and regulatory disclosure pages (static, must be pre-generated). Using a single rendering strategy served none of these well.

### Angular 18 Solution
Use **route-level render mode** to assign the optimal strategy per route.

```typescript
// app.routes.server.ts
import { RenderMode, ServerRoute } from '@angular/ssr';

export const serverRoutes: ServerRoute[] = [
  // Marketing + compliance — prerendered at build time (static HTML, CDN-cached)
  { path: '',              renderMode: RenderMode.Prerender },
  { path: 'pricing',       renderMode: RenderMode.Prerender },
  { path: 'about',         renderMode: RenderMode.Prerender },
  { path: 'disclosures',   renderMode: RenderMode.Prerender },
  { path: 'disclosures/:id', renderMode: RenderMode.Prerender,
    async getPrerenderParams() {
      // Pre-generate all regulatory disclosure pages
      return disclosureIds.map(id => ({ id }));
    }
  },

  // Market data pages — SSR on each request (live prices)
  { path: 'markets',          renderMode: RenderMode.Server },
  { path: 'markets/:symbol',  renderMode: RenderMode.Server },

  // Authenticated portfolio — CSR only (never SSR private data)
  { path: 'portfolio',    renderMode: RenderMode.Client },
  { path: 'transactions', renderMode: RenderMode.Client },
  { path: 'settings',     renderMode: RenderMode.Client },
];
```

### Impact

| Route Group | Strategy | Result |
|---|---|---|
| Marketing / Disclosures | Prerender | SEO score 100, CDN-cached, compliance-safe |
| Markets data | Server | Live prices, crawlable |
| Portfolio / Transactions | Client | Private data never exposed in SSR |
| Overall TTFB | — | Reduced from 420ms → 35ms (static routes from CDN) |

---

## Use Cases Summary

| # | Scenario | Key Feature Used | Primary Benefit |
|---|---|---|---|
| 1 | E-Commerce Cart | Zoneless + Signals | 3x fewer re-renders, -13KB bundle |
| 2 | Healthcare Form | `@let` template variables | 40% less template code |
| 3 | News Media SSR | Incremental Hydration | TTI 5.8s → 1.9s |
| 4 | SaaS Dashboard | `linkedSignal()` | Eliminated complex state sync logic |
| 5 | Fintech App | Route-level Render Mode | Per-route optimal SEO + security |

# Section 4 — Interview Q&A

---

## Basic Questions (5)

---

**Q1. What are the three core signal APIs that became stable in Angular 18?**

**Answer:**
The three core signal APIs promoted to **stable** in Angular 18 are:

1. **`signal(initialValue)`** — Creates a writable reactive state container
2. **`computed(fn)`** — Creates a lazily evaluated, read-only derived signal
3. **`effect(fn)`** — Registers a side effect that runs whenever its tracked signals change

```typescript
import { signal, computed, effect } from '@angular/core';

const price = signal(100);
const taxed = computed(() => price() * 1.08);

effect(() => console.log('Price updated:', price()));

price.set(200); // effect runs, taxed recalculates
```

Being **stable** means: production-safe, no breaking changes without a major version bump.

---

**Q2. What is Zoneless Change Detection in Angular 18 and how do you enable it?**

**Answer:**
Zoneless Change Detection is an **experimental** feature that removes Zone.js from the Angular application entirely. Without Zone.js, Angular only re-renders when:
- A **signal** value changes
- An **`async` pipe** emits a new value
- `ChangeDetectorRef.markForCheck()` is called manually

**Enable it:**
```typescript
// main.ts
import { provideExperimentalZonelessChangeDetection } from '@angular/core';

bootstrapApplication(AppComponent, {
  providers: [provideExperimentalZonelessChangeDetection()]
});
```

Also remove `zone.js` from `polyfills` in `angular.json`.

**Benefits:** Smaller bundle (-13KB), predictable rendering, better performance.

---

**Q3. What is the difference between `computed()` and `linkedSignal()`?**

**Answer:**

| Feature | `computed()` | `linkedSignal()` |
|---|---|---|
| Writable | ❌ No | ✅ Yes |
| Derived from source | ✅ Yes | ✅ Yes |
| Resets when source changes | ✅ Always | ✅ Yes |
| Can be manually overridden | ❌ No | ✅ Yes |
| Best for | Read-only derived data | UI selection state |

```typescript
const items = signal(['A', 'B', 'C']);
const selected = linkedSignal(() => items()[0]); // default: 'A'

selected.set('B');       // manual override
console.log(selected()); // 'B'

items.set(['X', 'Y']);   // source changes → resets
console.log(selected()); // 'X' (back to derived value)
```

---

**Q4. What is `@let` in Angular 18 templates and what problem does it solve?**

**Answer:**
`@let` is a new template syntax (Developer Preview) for declaring **local read-only variables** inside Angular templates.

**Problem it solves:** The `*ngIf as` workaround was the only way to create template-local variables, but it introduced structural directive side effects and required an extra DOM wrapper.

```html
<!-- OLD: Workaround -->
<ng-container *ngIf="user$ | async as user">
  <p>{{ user.name }} — {{ user.email }}</p>
</ng-container>

<!-- NEW: Clean @let -->
@let user = user$ | async;
@let fullName = user?.firstName + ' ' + user?.lastName;
@if (user) {
  <p>{{ fullName }} — {{ user.email }}</p>
}
```

**Key rules:** `@let` is block-scoped, read-only, and can reference signals, async pipes, and expressions.

---

**Q5. What is the `output()` function and how does it replace `@Output()`?**

**Answer:**
`output()` is a new function-based API (Developer Preview in Angular 18) for declaring component outputs, replacing the `@Output()` decorator + `EventEmitter` pattern.

```typescript
// BEFORE
import { Output, EventEmitter } from '@angular/core';
@Output() clicked = new EventEmitter<string>();

// AFTER
import { output } from '@angular/core';
clicked = output<string>();
```

**Differences:**
- No `EventEmitter` import needed
- Returns `OutputEmitterRef<T>` instead of `EventEmitter<T>`
- More consistent with `input()` signal API style
- Template usage `(clicked)="handler($event)"` is **identical** — no parent changes needed

---

## Intermediate Questions (5)

---

**Q6. What are the valid triggers for `@defer (hydrate on ...)` in incremental hydration?**

**Answer:**
Incremental hydration (Developer Preview in Angular 18) supports the following `hydrate on` triggers:

| Trigger | Description |
|---|---|
| `hydrate on viewport` | Hydrates when element enters the browser viewport |
| `hydrate on interaction` | Hydrates on first user interaction (click, focus, keydown) |
| `hydrate on idle` | Hydrates when browser `requestIdleCallback` fires |
| `hydrate on timer(2000)` | Hydrates after a fixed delay (ms) |
| `hydrate when condition` | Hydrates when a boolean expression becomes `true` |

```html
@defer (hydrate on viewport) {
  <app-product-recommendations />
}

@defer (hydrate when isLoggedIn()) {
  <app-user-dashboard />
}
```

Setup requires: `provideClientHydration(withIncrementalHydration())` in `app.config.ts`.

---

**Q7. What are the three `RenderMode` options and when should you use each?**

**Answer:**

| Mode | Import | When to Use |
|---|---|---|
| `RenderMode.Prerender` | `@angular/ssr` | Static content known at build time (blog, pricing, docs) |
| `RenderMode.Server` | `@angular/ssr` | Dynamic personalized pages (product listings with live data) |
| `RenderMode.Client` | `@angular/ssr` | Authenticated/private pages that should never be SSR'd |

```typescript
// app.routes.server.ts
export const serverRoutes: ServerRoute[] = [
  { path: 'blog/:slug', renderMode: RenderMode.Prerender,
    async getPrerenderParams() { return slugList.map(slug => ({ slug })); }
  },
  { path: 'shop',      renderMode: RenderMode.Server },
  { path: 'dashboard', renderMode: RenderMode.Client },
];
```

---

**Q8. How do signals interact with Angular's change detection in a zoneless app?**

**Answer:**
In a zoneless app, Angular's scheduler uses the **signal graph** to determine what to re-render:

1. When `signal.set()` or `signal.update()` is called, Angular marks the signal as **dirty**
2. Angular schedules a microtask to check which components read that signal
3. Only components that **read the changed signal** in their template are re-rendered
4. `computed()` values are lazily recalculated only when read again

```typescript
// Only the component reading `count` re-renders — not siblings or parents
const count = signal(0);

@Component({
  template: `<p>{{ count() }}</p>`  // This component subscribes to count
})
export class CounterComponent {
  count = inject(CounterStore).count;
}
```

This is fundamentally different from Zone.js which triggers `ApplicationRef.tick()` (full tree check) on every async event.

---

**Q9. How do you write unit tests for components using `output()`?**

**Answer:**
The `output()` API returns an `OutputEmitterRef` which has a `.subscribe()` method for testing.

```typescript
import { TestBed } from '@angular/core/testing';
import { outputToObservable } from '@angular/core/rxjs-interop';
import { RatingComponent } from './rating.component';

describe('RatingComponent', () => {
  it('should emit ratingSubmitted when submit is clicked', () => {
    const fixture = TestBed.createComponent(RatingComponent);
    const component = fixture.componentInstance;
    const emitted: number[] = [];

    // Subscribe to output
    const sub = outputToObservable(component.ratingSubmitted)
      .subscribe(v => emitted.push(v));

    component.selectedStar.set(4);
    component.submitRating();

    expect(emitted).toEqual([4]);
    sub.unsubscribe();
  });
});
```

---

**Q10. What is `input.required()` and how does it differ from `input()`?**

**Answer:**
`input.required<T>()` is a **stable** signal input (stable in Angular 18) that declares an input with **no default value** — Angular will throw an error at development time if the parent does not provide a value.

```typescript
// Required input — must be provided by parent
productId = input.required<string>();

// Optional input — has a default value
label = input('Default Label');
label = input<string | undefined>(undefined);

// Using required input in template/computed
productUrl = computed(() => `/products/${this.productId()}`);
```

**Error without value:**
```
NG0950: Required input 'productId' from component 'ProductCardComponent' must be specified.
```

Unlike `@Input({ required: true })` which was a runtime check in Angular 16, `input.required()` integrates with the type system — the return type is `InputSignal<T>` (not `InputSignal<T | undefined>`).

---

## Advanced Questions (5)

---

**Q11. Explain the signal graph and how Angular avoids unnecessary re-computation.**

**Answer:**
Angular maintains a **reactive graph** (directed acyclic graph) of signal dependencies:

- Each `signal()` is a **source node**
- Each `computed()` is a **derived node** that tracks which signals it reads
- Each component template is also a **consumer** that tracks signal reads during rendering

**Lazy evaluation:** `computed()` values are not recalculated immediately when a dependency changes — they are marked **stale**. The recalculation happens only when the computed signal is **read** again.

**Glitch-free:** If multiple signals change in the same synchronous block, Angular batches the re-renders — no intermediate states are rendered.

```typescript
const a = signal(1);
const b = signal(2);
const c = computed(() => a() + b());    // depends on a, b
const d = computed(() => c() * 2);     // depends on c (transitively a, b)

// Batch update — d is only recalculated once, not twice
a.set(10);
b.set(20);
// One re-render: d() = (10 + 20) * 2 = 60
```

---

**Q12. What are the trade-offs of using Zoneless Change Detection in Angular 18?**

**Answer:**

**Advantages:**
- ~13KB bundle reduction (zone.js removed)
- Predictable, targeted re-renders
- Easier debugging (no accidental re-renders from setTimeout/XHR patches)
- Aligns with the long-term Angular direction

**Disadvantages / Trade-offs:**

| Challenge | Explanation |
|---|---|
| Legacy code incompatibility | Third-party libraries using Zone.js-patched APIs may not trigger detection |
| Manual `markForCheck()` needed | RxJS Observables not converted with `toSignal()` need manual detection |
| Still experimental | API may change before stable release |
| Team learning curve | All developers must understand signal-based reactivity |

**Migration strategy:** Start with `OnPush` components (already minimal Zone.js reliance) → convert to signals → enable zoneless per-module → finally remove zone.js globally.

---

**Q13. How does `linkedSignal()` differ from manually resetting a signal in `effect()`?**

**Answer:**
Both approaches can achieve similar results, but `linkedSignal()` is superior:

**Manual approach with `effect()` (problematic):**
```typescript
// BAD: Creates circular dependency risk + verbose code
const items = signal<Item[]>([]);
const selectedItem = signal<Item | null>(null);

effect(() => {
  // This is fragile — effect writes to selectedItem while reading items
  selectedItem.set(items()[0] ?? null);
});
```

**`linkedSignal()` approach (correct):**
```typescript
// GOOD: Clean, no circular dependency, writable
const items = signal<Item[]>([]);
const selectedItem = linkedSignal(() => items()[0] ?? null);
```

**Key differences:**
- `effect()` writing to signals can cause circular dependency errors — Angular will throw a `NG0600` error
- `linkedSignal()` is specifically designed for this "reset-but-overridable" pattern
- `linkedSignal()` is synchronous (no async scheduling like `effect()`)
- `linkedSignal()` preserves writability — the user can still override the value

---

**Q14. How do you implement Material 3 theming with dynamic color in Angular 18?**

**Answer:**
Angular Material 18 M3 theming uses a **token-based system** with Sass mixins and CSS custom properties.

```scss
// styles.scss
@use '@angular/material' as mat;

// Define an M3 theme
$light-theme: mat.define-theme((
  color: (
    theme-type: light,
    primary: mat.$violet-palette,
    tertiary: mat.$orange-palette,
  ),
  typography: (
    brand-family: 'Google Sans, sans-serif',
    plain-family: 'Roboto, sans-serif',
    bold-weight: 700,
  ),
  density: (scale: 0)
));

$dark-theme: mat.define-theme((
  color: (
    theme-type: dark,
    primary: mat.$violet-palette,
  )
));

// Apply base theme
html { @include mat.all-component-themes($light-theme); }

// Dark mode support
@media (prefers-color-scheme: dark) {
  html { @include mat.all-component-color-themes($dark-theme); }
}

// Component-level theme override
.compact-ui {
  @include mat.button-density(-3);
}
```

**Dynamic theming at runtime:**
```typescript
// theme.service.ts — switch between themes at runtime
@Injectable({ providedIn: 'root' })
export class ThemeService {
  setDarkMode(enabled: boolean) {
    document.documentElement.classList.toggle('dark-theme', enabled);
  }
}
```

---

**Q15. What happens to existing `@Output()` / `EventEmitter` code in Angular 18 — is migration required?**

**Answer:**
**No — migration is NOT required.** `@Output()` with `EventEmitter` continues to work in Angular 18 and beyond. The new `output()` API is additive and runs alongside the old API.

**Migration is optional and can be done incrementally:**

```typescript
// Step 1: Old code (still works perfectly in v18)
@Output() productSelected = new EventEmitter<Product>();

// Step 2: New equivalent (same template syntax, no parent changes)
productSelected = output<Product>();
```

**When to prefer `output()`:**
- New components being written from scratch
- Components already using `input()` signal inputs (consistency)
- When you want to avoid `EventEmitter` RxJS overhead

**Angular provides a migration schematic:**
```bash
ng generate @angular/core:output-migration
```

---

## Scenario-Based Questions (5)

---

**Q16. Your team is migrating a large Angular 14 app to Angular 18 with zoneless. What migration path do you recommend?**

**Answer:**
A phased migration approach:

**Phase 1 — Upgrade to Angular 18 (keep Zone.js)**
```bash
ng update @angular/core@18 @angular/cli@18
```
Run all tests — fix any breaking changes.

**Phase 2 — Enable `OnPush` everywhere**
```typescript
@Component({ changeDetection: ChangeDetectionStrategy.OnPush })
```
This limits Zone.js reliance to explicit triggers.

**Phase 3 — Convert state to Signals**
```typescript
// Convert BehaviorSubject to signal
// BEFORE: private subject = new BehaviorSubject(0);
// AFTER:
count = signal(0);
```

**Phase 4 — Enable zoneless experimentally**
```typescript
// Test in a feature branch first
provideExperimentalZonelessChangeDetection()
```

**Phase 5 — Fix remaining issues**
- Replace `setTimeout`-based code with signals/effects
- Convert remaining Observable flows to `toSignal()`
- Remove zone.js from `polyfills`

**Phase 6 — Full zoneless in production**
Full testing suite must pass before removing zone.js completely.

---

**Q17. A user selects a product variant (color/size) in a shopping cart. When they change the product entirely, the selection should reset to the first available variant. How do you implement this cleanly in Angular 18?**

**Answer:**
This is the perfect use case for `linkedSignal()`:

```typescript
@Component({
  selector: 'app-product-detail',
  standalone: true,
  template: `
    <select (change)="selectedProduct.set(getProduct($any($event.target).value))">
      @for (p of products; track p.id) {
        <option [value]="p.id">{{ p.name }}</option>
      }
    </select>

    <div class="variants">
      @for (variant of selectedProduct().variants; track variant.id) {
        <button
          [class.selected]="selectedVariant()?.id === variant.id"
          (click)="selectedVariant.set(variant)">
          {{ variant.color }} / {{ variant.size }}
        </button>
      }
    </div>

    <p>Selected: {{ selectedVariant()?.color }} {{ selectedVariant()?.size }}</p>
  `
})
export class ProductDetailComponent {
  products = inject(ProductService).getAll();
  selectedProduct = signal(this.products[0]);

  // Auto-resets to first variant when product changes
  // But user can click to override
  selectedVariant = linkedSignal(() => this.selectedProduct().variants[0]);
}
```

---

**Q18. Your Angular 18 SSR app has a route `/reports/:id` with 500+ report IDs. How do you prerender only the top 10 most-accessed reports and SSR the rest on demand?**

**Answer:**
Use `RenderMode.Prerender` with `getPrerenderParams()` for the top 10, and a separate `RenderMode.Server` fallback:

```typescript
// app.routes.server.ts
import { RenderMode, ServerRoute } from '@angular/ssr';
import { inject } from '@angular/core';
import { ReportService } from './report.service';

export const serverRoutes: ServerRoute[] = [
  {
    path: 'reports/:id',
    renderMode: RenderMode.Prerender,
    async getPrerenderParams() {
      // Called at build time — fetch top 10 report IDs from API or config
      const topIds = ['rpt-001', 'rpt-002', 'rpt-003', /* ... top 10 */];
      return topIds.map(id => ({ id }));
      // All other IDs not in this list will be handled by Server mode below
    },
    fallback: RenderMode.Server  // Unknown IDs fall back to SSR
  },
];
```

**Result:** Top 10 reports served as static HTML from CDN in <10ms. All other reports SSR'd on demand in ~120ms. No CSR penalty for any report.

---

**Q19. You're building a fintech app where portfolio data must never be included in SSR responses. How do you enforce this with Angular 18?**

**Answer:**
Use `RenderMode.Client` for all authenticated/private routes:

```typescript
// app.routes.server.ts
export const serverRoutes: ServerRoute[] = [
  // Public pages — SSR/Prerender is fine
  { path: '',         renderMode: RenderMode.Prerender },
  { path: 'markets',  renderMode: RenderMode.Server },

  // ALL private routes — Client-side only
  // Server never renders these — no risk of private data in HTML/HTTP logs
  { path: 'portfolio',        renderMode: RenderMode.Client },
  { path: 'portfolio/:id',    renderMode: RenderMode.Client },
  { path: 'transactions',     renderMode: RenderMode.Client },
  { path: 'account',          renderMode: RenderMode.Client },
  { path: 'account/settings', renderMode: RenderMode.Client },
  { path: 'tax-reports',      renderMode: RenderMode.Client },
];
```

**Additional security layers:**
- Route guards checking auth tokens before component renders
- `isPlatformServer()` checks in services to prevent server-side API calls with sensitive tokens
- HTTP security headers (HSTS, CSP) via Angular SSR middleware

---

**Q20. A junior developer asks: "Should I use Signals or RxJS Observables in my new Angular 18 component?" What advice do you give?**

**Answer:**
The answer is **"both, for different purposes."** They solve different problems:

**Use Signals for:**
- Local component state (`count`, `isOpen`, `selectedItem`)
- Derived/computed values (`total`, `isValid`, `filteredList`)
- Cross-component shared state (in a signal-based service)
- Anything that drives template rendering

**Use RxJS Observables for:**
- HTTP requests (`HttpClient` returns Observables)
- WebSocket / SSE streams
- Complex async flows (`switchMap`, `mergeMap`, `debounceTime`)
- Event streams that need operators (`fromEvent`, `interval`)

**Bridge between them:**
```typescript
// Observable → Signal (for templates)
user = toSignal(this.http.get<User>('/api/user'));

// Signal → Observable (for RxJS operators)
search$   = toObservable(this.searchQuery);
results$  = this.search$.pipe(
  debounceTime(300),
  switchMap(q => this.http.get(`/api/search?q=${q}`))
);
resultsSignal = toSignal(this.results$, { initialValue: [] });
```

**Rule of thumb:** "Start with signals. Reach for RxJS when you need async coordination operators."

---

## Quick Reference Cheatsheet

| Feature | Angular 17 | Angular 18 |
|---|---|---|
| `signal()` / `computed()` / `effect()` | Developer Preview | ✅ **Stable** |
| `input()` / `input.required()` | Developer Preview | ✅ **Stable** |
| `output()` | ❌ Not available | Developer Preview |
| `linkedSignal()` | ❌ Not available | Developer Preview |
| `@let` template variable | ❌ Not available | Developer Preview |
| Zoneless Change Detection | ❌ Not available | Experimental |
| Incremental Hydration | ❌ Not available | Developer Preview |
| Route-level Render Mode | ❌ Not available | Developer Preview |
| Angular Material 3 (M3) | Developer Preview | ✅ **Stable** |
| TypeScript version | 5.2 | 5.4 |

---

## Before / After Migration Table

| Pattern | Before (Angular 17) | After (Angular 18) |
|---|---|---|
| Reactive state | `BehaviorSubject` / `@Input()` | `signal()` (stable) |
| Derived state | `combineLatest` / getter | `computed()` (stable) |
| Side effects | `ngOnChanges` / tap | `effect()` (stable) |
| Component output | `@Output() clicked = new EventEmitter()` | `clicked = output()` |
| Selection state | `ngOnChanges` reset logic | `linkedSignal()` |
| Template local var | `*ngIf="obs$ \| async as x"` | `@let x = obs$ \| async` |
| Change detection | Zone.js (always on) | Zoneless (opt-in experimental) |
| SSR strategy | Global (all routes same) | Per-route `RenderMode` |
| Material design | M2 (experimental M3) | M3 **Stable** |